# PatchCore baseline and training-data contamination study

[PatchCore](https://arxiv.org/abs/2106.08265) (memory bank of pretrained patch features + coreset subsampling +
nearest-neighbour scoring) was used as the strong baseline for surface-defect detection on production data,
and to answer a practical question: **how sensitive is it to defective or ambiguous images slipping into the
"good" training set?**

The datasets are proprietary (anonymised as *Product A/B/C*), so only the commands and the logged metrics are kept.

In [ ]:
# Upstream PatchCore implementation (Roth et al., CVPR 2022).
# The experiments were run from a personal fork of this repository with minor I/O changes.
!git clone https://github.com/amazon-science/patchcore-inspection.git
%cd patchcore-inspection
%pip install -q faiss-gpu-cu12
!python setup.py -q install

In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

## 1. Baseline runs on the anonymised production datasets

Each dataset follows the MVTec AD folder layout (`train/good`, `test/good`, `test/<defect>`).
Backbone: WideResNet-50 (layers 2+3), patch size 5, 10 % greedy coreset, 1-NN anomaly scoring.

### Product A — area-scan camera images

In [ ]:
!python bin/run_patchcore.py --save_patchcore_model --gpu 0 --seed 0 --log_group strict_labels --log_project patchcore_results {DATA_ROOT}/handpicked/resultsproduct_a patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d product_a mvtec {DATA_ROOT}/handpicked

instance_auroc: 0.943
optimal_threshold: 0.123
accuracy_wrt_optimal_threshold: 0.918


### Product A — line-scan (CIS) camera images

In [ ]:
!python bin/run_patchcore.py --save_segmentation_images --save_patchcore_model --gpu 0 --seed 0 --log_group product_a_line_scan --log_project patchcore_results {DATA_ROOT}/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d product_a_line_scan mvtec {DATA_ROOT}/handpicked

instance_auroc: 0.993
optimal_threshold: 0.272
accuracy_wrt_optimal_threshold: 0.962


### Product A — line-scan images, downscaled to 32x32 (edge-deployment feasibility test)

In [ ]:
!python bin/run_patchcore.py --save_segmentation_images --save_patchcore_model --gpu 0 --seed 0 --log_group product_a_line_scan32 --log_project patchcore_results {DATA_ROOT}/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 32 --imagesize 32 -d product_a_line_scan mvtec {DATA_ROOT}/handpicked

instance_auroc: 0.995
optimal_threshold: 0.154
accuracy_wrt_optimal_threshold: 0.981


### Product C — line-scan images (hardest case: subtle, low-contrast defects)

In [ ]:
!python bin/run_patchcore.py --save_segmentation_images --save_patchcore_model --gpu 0 --seed 0 --log_group product_ccis --log_project patchcore_results {DATA_ROOT}/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d base128 mvtec {DATA_ROOT}/product_c

instance_auroc: 0.620
optimal_threshold: 0.093
accuracy_wrt_optimal_threshold: 0.505


## 2. Robustness to contaminated training data

In production, the "good" training set is never perfectly clean. Here the 50- and 200-image training sets are
deliberately polluted with *ambiguous* or *defective* samples (10 % and 20 %) to measure how much the
image-level AUROC degrades.

### 50 clean training images (reference)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group base50 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d base50 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.996
optimal_threshold: 0.320
accuracy_wrt_optimal_threshold: 0.953


### 45 clean + 5 ambiguous (10 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group ambig10percentin50 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d ambig10percentin50 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.996
optimal_threshold: 0.209
accuracy_wrt_optimal_threshold: 0.972


### 40 clean + 10 ambiguous (20 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group ambig20percentin50 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d ambig20percentin50 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.996
optimal_threshold: 0.267
accuracy_wrt_optimal_threshold: 0.972


### 45 clean + 5 defective (10 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group defective10percentin50 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d defective10percentin50 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.996
optimal_threshold: 0.198
accuracy_wrt_optimal_threshold: 0.937


### 40 clean + 10 defective (20 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group defective20percentin50 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d defective20percentin50 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.995
optimal_threshold: 0.334
accuracy_wrt_optimal_threshold: 0.956


### 200 clean training images (reference)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group base200 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d base200 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.993
optimal_threshold: 0.316
accuracy_wrt_optimal_threshold: 0.962


### 180 clean + 20 ambiguous (10 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group ambig10percentin200 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d ambig10percentin200 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.993
optimal_threshold: 0.329
accuracy_wrt_optimal_threshold: 0.975


### 160 clean + 40 ambiguous (20 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group ambig20percentin200 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d ambig20percentin200 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.993
optimal_threshold: 0.344
accuracy_wrt_optimal_threshold: 0.969


### 180 clean + 20 defective (10 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group defective10percentin200 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d defective10percentin200 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.989
optimal_threshold: 0.405
accuracy_wrt_optimal_threshold: 0.956


### 160 clean + 40 defective (20 %)

In [ ]:
!python bin/run_patchcore.py --gpu 0 --seed 0 --log_group defective20percentin200 --log_project patchcore_results {DATA_ROOT}/product_a/patchcore_results patch_core -b wideresnet50 -le layer2 -le layer3 --faiss_on_gpu --pretrain_embed_dimension 1024  --target_embed_dimension 2048 --anomaly_scorer_num_nn 1 --patchsize 5 sampler -p 0.1 approx_greedy_coreset dataset --resize 224 --imagesize 224 -d defective20percentin200 mvtec {DATA_ROOT}/product_a/handpicked

instance_auroc: 0.988
optimal_threshold: 0.378
accuracy_wrt_optimal_threshold: 0.969
